# Makemore part 2: Multi Layer Perceptron approach

This notebook contains my notes (ah, [hello](www.bazcuervo.net)) from Karpathy's [second lesson](https://www.youtube.com/watch?v=TCH_1BHY58I) from [*Neural Networks: Zero to Hero*](https://www.youtube.com/playlist?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ).

It's more exhaustive than the [notebook](https://github.com/karpathy/nn-zero-to-hero/blob/master/lectures/makemore/makemore_part2_mlp.ipynb) that he has uploaded, given that here I reproduce all the steps in the tutorial instead of just building the final result (the process is quite iterative).

## INTRO

We'll be following the approach introduced in [A Neural Probabilistic Language Model](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf), in which instead of going from the purely statistical approach to predicting words / characters, they follow an embedding + MLP approach. It's not the first paper that introduced that idea, but it's one of the most cited and populars when it comes to it. 

In the paper they use **words instead of characters**, but we'll analogously apply this approach with characters. 

## Load modules, data and build character-index mappings

In [1]:
import torch

%matplotlib inline

We'll use the same dataset of names as in all of the other lessons.

In [2]:
words = open("../data/names.txt", "r").read().splitlines()

In [3]:
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
len(words)

32033

As in the previous lesson, we'll build the mappings of character-index and index-character. This is the way of working with characters numerically.

In [5]:
unique_characters: list[str] = sorted(list(set("".join(words))))
# character_to_index
ctoi = {c: i + 1 for i, c in enumerate(unique_characters)}
# for the special value "." (beginnings and ends of words) we'll assign the integer 0
ctoi["."] = 0
# and the reverse counterpart, to be able to get the corresponding character to a given index
itoc = {i: c for c, i in ctoi.items()}

In [6]:
ctoi

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [7]:
itoc

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

## Build the dataset

Now that we have the words, we have to build our dataset: the `X` and `Y` tensors that will make the input-output pairs, that will let us train out NN and get the proper weights.

In [8]:
# context length: how many characters do we take to predict the next one?
block_size = 3
X, Y = [], []

for w in words[:4]:
    print(w)
    # first we fill the context with all zeros (the character ".")
    context = [0] * block_size

    # we iterate over each character in the given word
    for ch in w + ".":
        # we get the index of the character
        ix = ctoi[ch]
        # and given the existing context that we have (i.e. previous characters that we read in this word),
        # we use that pair context-character to create an X-Y value in our dataset (input-output)
        X.append(context)
        # (remember that context is a set of indexes as well)
        Y.append(ix)
        # for printing, we convert them into characters instead of indexes
        print("".join(itoc[i] for i in context), "--->", itoc[ix])
        # finally, we append this given character to the context and slice that context so that its size
        # is equal to the block size we specified
        context = context[1:] + [ix]

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .


So up until now it's exactly the same that we were doing before, but instead of using bigrams (`block_size = 2`) we're using sets of $x_i$ (that will predict an output $y_i$) of arbitrary length.

We'll do the same but with all the words:

In [9]:
# context length: how many characters do we take to predict the next one?
block_size = 3
X, Y = [], []

for w in words:
    # first we fill the context with all zeros (the character ".")
    context = [0] * block_size

    # we iterate over each character in the given word
    for ch in w + ".":
        # we get the index of the character
        ix = ctoi[ch]
        # and given the existing context that we have (i.e. previous characters that we read in this word),
        # we use that pair context-character to create an X-Y value in our dataset (input-output)
        X.append(context)
        # (remember that context is a set of indexes as well)
        Y.append(ix)
        # finally, we append this given character to the context and slice that context so that its size
        # is equal to the block size we specified
        context = context[1:] + [ix]

# and finally we  convert those lists into vectors, because that's what we use to work with matrices
X = torch.tensor(X)
Y = torch.tensor(Y)

In [10]:
print(X.shape, X.dtype)
print(Y.shape, Y.dtype)

torch.Size([228146, 3]) torch.int64
torch.Size([228146]) torch.int64


Viendo la forma que tienen estos tensores, tiene sentido.

In [11]:
X[:10]

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22]])

In [12]:
Y[:10]

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9])

## Embedding

An embedding gives the NN a **compact** and **learnable** representation of each character, even being able to establish similarities between characters. With the one hot encoding approach we were just fixing the representation, making it arbitrarily bigger and avoiding any possible learnability there.

Furthermore, you're separating the two concerns that we were mixing in our previous approach with one hot encoded vectors: the embedding learns how to *represent* those characters, and the first layer learns what to *do* with those characters.

Also, we have to take into account that the dimension of the vectors that we'll input into 